In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('UNSTRUCTURED_API_KEY')
if api_key:
	os.environ["UNSTRUCTURED_API_KEY"] = api_key
else:
	print("Warning: UNSTRUCTURED_API_KEY environment variable not set")

In [ ]:
import pickle
from pathlib import Path

cache_path = Path("../data/processed/docs_cache.pkl")

# Load from cache if exists
if cache_path.exists():
    with open(cache_path, 'rb') as f:
        docs = pickle.load(f)
    print("Loaded from cache")
else:
    # Load via API
    from langchain_unstructured import UnstructuredLoader
    loader = UnstructuredLoader(
        file_path="../data/raw/finetunexlmr.pdf",
        api_key=api_key,
        partition_via_api=True,
    )
    docs = loader.load()
    
    # Save to cache
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, 'wb') as f:
        pickle.dump(docs, f)
    print("Loaded from API and cached")

In [ ]:
import pickle
from pathlib import Path

# Save loaded documents
cache_path = Path("../data/processed/docs_cache.pkl")
cache_path.parent.mkdir(parents=True, exist_ok=True)

with open(cache_path, 'wb') as f:
    pickle.dump(docs, f)

print(f"Documents saved to {cache_path}")

In [ ]:
docs[4].page_content

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv('GROQ_API_KEY'),
    temperature=0
)

# response = llm.invoke("Explain RAG in one sentence.")
# print(response.content)


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

In [ ]:
# Merge tiny documents and split into chunks-
from langchain_core.documents import Document
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200

# Merge all non-empty page content
full_text = "\n".join([doc.page_content.strip() for doc in docs if doc.page_content.strip() != ""])
docs_to_split = Document(page_content=full_text)
print(f"Merged {len(docs)} docs into 1 document with {len(full_text)} characters")

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", "? ", "! ", "; ", " ", ""],
    length_function=len,
)

# Split the merged document
chunked_docs = text_splitter.split_documents([docs_to_split])
print(f"Split merged document into {len(chunked_docs)} chunks")

# Notes:
# - Merging first avoids creating too many tiny chunks for tiny documents
# - Larger chunks work better for similarity search (we usually select only a few chunks)
# - Splitting after merging creates manageable, meaningful pieces


In [ ]:
system_prompt = """"Instructions:
        1. Use ONLY the provided context to answer the user's query.
        2. Extract the answer strictly from the context. Do NOT infer, assume, or add missing details.
        3. Do NOT use external knowledge.
        4. You may rewrite the extracted content to improve grammar, clarity, and formatting, but DO NOT add new information."
        5. Present the answer in a clean, well-structured format using proper sentences and bullet points where necessary.
        6. If the answer is not explicitly present, reply exactly: \"The provided context does not contain this information.
"""

In [ ]:
from langchain_community.vectorstores import Qdrant

# Create vectorstore and add documents
vectorstore = Qdrant.from_documents(
    documents=chunked_docs,
    embedding=model,
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    collection_name="pdf_chunks",
    force_recreate=True,  # Recreate if exists
)

print(f"Added {len(chunked_docs)} documents to Qdrant!")

In [ ]:
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":2}
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "Context:{context} Input:{input}")
])

In [ ]:
from langchain_classic.chains import create_history_aware_retriever

retriever_prompt = ChatPromptTemplate.from_messages([
    ("system","Rewrite the user query using the chat history if needed. Do NOT answer."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

In [ ]:
history_aware_retriever = create_history_aware_retriever(
    llm,retriever,retriever_prompt
)

In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
question_answer_chain = create_stuff_documents_chain(
    llm,prompt=answer_prompt
)

In [ ]:
from langchain_core.runnables import RunnableParallel
rag_chain = (
    RunnableParallel({
        'input':lambda d: d['input'],
        'chat_history':lambda d: d['chat_history'],
        'context': lambda d:history_aware_retriever.invoke({
            'chat_history': d['chat_history'],
            'input': d['input'],
        })
    })
    | question_answer_chain
)

In [ ]:
query = 'What is the research about?'

search_results = retriever.invoke(query)


In [ ]:
chat_history=[]

In [ ]:
from langchain_core.messages import AIMessage,HumanMessage
response = rag_chain.invoke({
    'input':query,
    'chat_history':chat_history
})
chat_history.extend(
    [
        HumanMessage(content=query),
        AIMessage(content=response)
    ]
)

In [ ]:
response